In [ ]:
import Omniwheel_Protocol
import serial
import time

Arduino_ID = 0x20
REQUEST_ODOMETER = 0xA0
ANSWER_ODOMETER = 0xB0
ENCODER_CLEAR = 0xC2
MID_ENCODER_WHEEL_1 = 0x80
MID_ENCODER_WHEEL_2 = 0x81
MID_ENCODER_WHEEL_3 = 0x82
MID_list = [MID_ENCODER_WHEEL_1,
            MID_ENCODER_WHEEL_2,
            MID_ENCODER_WHEEL_3]
data_wheel_1_encoder_count = 0
data_wheel_2_encoder_count = 0
data_wheel_3_encoder_count = 0
send_packet = Omniwheel_Protocol.Packet()
recv_packet = Omniwheel_Protocol.Packet()
recv_packet.clearPacket()
send_packet.clearPacket()
recv_list = []
recv_parsing_packet = []
send_flag = False
Serial_Arduino = serial.Serial(port="/dev/ttyUSB0", baudrate=9600, timeout=.1)
time.sleep(1)
print("connect complete")

def Packet_send(_id, _cmd, _mid, _data=None):
    global send_flag
    if(send_flag == False):
        send_packet.clearPacket()
        send_packet.setID(_id)
        send_packet.setCMD(_cmd)
        send_packet.clearPayload()
        send_packet.addPayload(_mid, _data)
        send_packet.calcLRC_Lower()
        send_list = send_packet.packetToList()
        Serial_Arduino.write(send_list)
        send_flag = True
def Packet_receive(ser):
    global send_flag
    if(send_flag == True):
        while ser.inWaiting() > 0:
            Arduino_Data = ser.read(1)
            if(len(Arduino_Data) > 0):
                recv_list.append(ord(Arduino_Data))
                if(ord(Arduino_Data) == 0x03):
                    if(recv_packet.parsingList(recv_list)):
                        recv_parsing_packet.append(recv_packet)
                        recv_list.clear()
                        send_flag = False
                        break

def Received_packet():
    result = recv_parsing_packet[0]
    del recv_parsing_packet[0]
    return result


def Encoder_Data(packet):
    global MID_list
    global data_wheel_1_encoder_count
    global data_wheel_2_encoder_count
    global data_wheel_3_encoder_count

    packet_id = packet.getID()
    packet_cmd = packet.getCMD()

    if(packet_id == Arduino_ID):
        if(packet_cmd == ANSWER_ODOMETER):
            for payload in packet.getPayload():
                if(payload.getID() == MID_list[0]):
                    data_wheel_1_encoder_count = int(float(payload.getData()))
                elif(payload.getID() == MID_list[1]):
                    data_wheel_2_encoder_count = int(float(payload.getData()))
                elif(payload.getID() == MID_list[2]):
                    data_wheel_3_encoder_count = int(float(payload.getData()))

Packet_send(Arduino_ID, ENCODER_CLEAR, None)
send_flag = False
while(True):
    for mid in MID_list:
        Packet_send(Arduino_ID, REQUEST_ODOMETER, mid)
        time.sleep(0.1)
        Packet_receive(Serial_Arduino)
        if(len(recv_parsing_packet) > 0):
            p = Received_packet()
            Encoder_Data(p)
        print("X:"
            "+str(data_wheel_1_encoder_count)+"
            "+str(data_wheel_2_encoder_count)"+" Z: "+str(data_wheel_3_encoder_count))
        time.sleep(0.5)